# 02 · Três métodos de seleção e votação de features

**Etapas do fluxo:** 4 (três métodos de importância/seleção) e 5 (votação → features eleitas do ciclo).

| Método | O que mede | Agregação para o atributo original |
|---|---|---|
| `SelectKBest(mutual_info_classif)` | dependência estatística (inclusive não linear) entre atributo e alvo | maior score entre suas colunas one-hot |
| Importância do `RandomForestClassifier` | redução de impureza nas árvores | soma das importâncias das colunas one-hot |
| `RFE` com `LogisticRegression` | ordem de eliminação recursiva do atributo menos útil | melhor posição entre suas colunas one-hot |

**Regra de votação (definida antes, em `config.yaml`):** cada método aprova seu *top-k*; é eleito o atributo com
pelo menos `votos_minimos` aprovações; se forem eleitos menos que `min_features`, completa-se pelos não eleitos com mais
votos e, em empate, menor posição média nos três rankings. Tudo é ajustado **somente no treino** do ciclo.

In [1]:
import sys
import warnings
from pathlib import Path

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "config.yaml").exists())
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import json
import matplotlib.pyplot as plt
import pandas as pd

from src.config import caminho, carregar_config

pd.set_option("display.max_columns", 50)
cfg = carregar_config(RAIZ)

In [2]:
import mlflow
from src.ciclo import atualizar_contexto, carregar_contexto, diretorio_ciclo, obter_cycle_id, salvar_json
from src.features import executar_votacao
from src.rastreamento import run_etapa

cycle_id = obter_cycle_id(cfg)
contexto = carregar_contexto(cfg, cycle_id)
pasta_ciclo = diretorio_ciclo(cfg, cycle_id)
treino = pd.read_csv(caminho(cfg, cfg["dados"]["diretorio_processado"]) / "treino.csv")
print(cycle_id, contexto["data_version"], "| treino:", treino.shape)
print("Regra:", json.dumps({k: v for k, v in cfg["votacao"].items() if k != "metodos"}, ensure_ascii=False))

c20261123T184307Z-gh45241187784 v005-6563af51 | treino: (29617, 25)
Regra: {"top_k": 10, "votos_minimos": 2, "min_features": 6, "desempate": "ranking_medio"}


In [3]:
resultado = executar_votacao(treino, cfg)
for metodo, ranking in resultado["resultados"].items():
    print(f"\n=== {metodo} (top {cfg['votacao']['top_k']}) ===")
    display(ranking.head(cfg["votacao"]["top_k"]).round(4))


=== selectkbest (top 10) ===


,pontuacao,posicao,aprovada
feature,,,
euribor3m,0.0748,1,True
cons.price.idx,0.0705,2,True
cons.conf.idx,0.0686,3,True
nr.employed,0.0619,4,True
emp.var.rate,0.0545,5,True
contatado_antes,0.0363,6,True
poutcome,0.0286,7,True
previous,0.0222,8,True
pdays_dias,0.0196,9,True



=== random_forest (top 10) ===


,pontuacao,posicao,aprovada
feature,,,
nr.employed,0.1783,1,True
euribor3m,0.1435,2,True
emp.var.rate,0.1185,3,True
contatado_antes,0.1036,4,True
month,0.0649,5,True
cons.conf.idx,0.0602,6,True
cons.price.idx,0.0528,7,True
poutcome,0.0475,8,True
contact,0.0409,9,True



=== rfe (top 10) ===


,pontuacao,posicao,aprovada
feature,,,
contatado_antes,-1.0,1,True
month,-2.0,2,True
emp.var.rate,-4.0,3,True
cons.price.idx,-8.0,4,True
euribor3m,-9.0,5,True
contact,-11.0,6,True
poutcome,-12.0,7,True
faixa_etaria,-13.0,8,True
job,-16.0,9,True


## Tabela de votos e conjunto eleito

In [4]:
tabela_votos = resultado["tabela_votos"]
eleitas = resultado["eleitas"]
display(tabela_votos)
print(f"Features eleitas ({len(eleitas)}):", eleitas)

cores = ["tab:green" if e else "tab:gray" for e in tabela_votos["eleita"]]
tabela_votos["total_votos"].iloc[::-1].plot.barh(color=cores[::-1], figsize=(8, 6), title="Votos por atributo (verde = eleito)")
plt.axvline(cfg["votacao"]["votos_minimos"] - 0.5, color="red", linestyle="--")
plt.show()

,posicao_selectkbest,voto_selectkbest,posicao_random_forest,voto_random_forest,posicao_rfe,voto_rfe,total_votos,posicao_media,eleita,criterio
feature,,,,,,,,,,
euribor3m,1,1,2,1,5,1,3,2.67,True,votos
contatado_antes,6,1,4,1,1,1,3,3.67,True,votos
emp.var.rate,5,1,3,1,3,1,3,3.67,True,votos
cons.price.idx,2,1,7,1,4,1,3,4.33,True,votos
poutcome,7,1,8,1,7,1,3,7.33,True,votos
contact,10,1,9,1,6,1,3,8.33,True,votos
nr.employed,4,1,1,1,15,0,2,6.67,True,votos
month,14,0,5,1,2,1,2,7.00,True,votos
cons.conf.idx,3,1,6,1,14,0,2,7.67,True,votos


Features eleitas (9): ['euribor3m', 'contatado_antes', 'emp.var.rate', 'cons.price.idx', 'poutcome', 'contact', 'nr.employed', 'month', 'cons.conf.idx']


In [5]:
tabela_votos.to_csv(pasta_ciclo / "votacao.csv")
for metodo, ranking in resultado["resultados"].items():
    ranking.to_csv(pasta_ciclo / f"ranking_{metodo}.csv")
salvar_json(pasta_ciclo / "features_eleitas.json", {"cycle_id": cycle_id, "data_version": contexto["data_version"], "features": eleitas})

with run_etapa(cfg, contexto, "votacao_features"):
    votacao = cfg["votacao"]
    mlflow.log_params({
        "metodo_votacao": f">= {votacao['votos_minimos']} de 3 votos no top-{votacao['top_k']}",
        "top_k": votacao["top_k"], "votos_minimos": votacao["votos_minimos"], "min_features": votacao["min_features"],
        "desempate": votacao["desempate"], "metodos": "selectkbest,random_forest,rfe",
        "features_eleitas": ",".join(eleitas),
    })
    mlflow.log_metrics({"n_features_eleitas": len(eleitas), "n_features_candidatas": len(tabela_votos), "n_colunas_transformadas": resultado["n_colunas_transformadas"]})
    for arquivo in ["votacao.csv", "features_eleitas.json", *[f"ranking_{m}.csv" for m in resultado["resultados"]]]:
        mlflow.log_artifact(str(pasta_ciclo / arquivo))

contexto = atualizar_contexto(cfg, cycle_id, features_eleitas=eleitas)